# Testing Script

In [1]:
import os
import torch
import numpy as np
import plotly.express as px

from umap import UMAP

from captum.attr import Saliency

from torch_brain.utils import seed_everything

seed_everything(0)

base_dir = os.getcwd().split("notebooks")[0]
device = "callable" if torch.cuda.is_available() else "cpu"

print(f"Using: {device}")

Using: cpu


In [2]:
from neural_decode.dataset.transformer_dataloader import get_train_val_loaders

In [3]:
path_to_neural_dataset = os.path.join(base_dir, "data", "perich_miller_population_2018", "t_20130819_center_out_reaching")
train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(path_to_neural_dataset, batch_size=64)

num_units = len(train_dataset.get_unit_ids())
print(f"Num Units in Session: {num_units}")

Num Units in Session: 55


In [4]:
from neural_decode.models.transformer import TransformerNeuralDecoder 

In [5]:
# 2. Initialize Model
tf_model = TransformerNeuralDecoder(
    num_units=num_units,    # Num. of units inputted (spiking activity)
    #
    bin_size=10e-3,         # Duration (s) of bins
    sequence_length=1.0,    # Context length of the model
    #
    dim_output=2,           # Output dimension of final readout layer
    dim_hidden=128,         # Hidden dimension of the model
    n_layers=3,             # Num. of transformer layers
    n_heads=4,
).to(device)

In [6]:
saved_model = torch.load(os.path.join(base_dir, "results", "models", "transformer_model.pth"), map_location=torch.device('cpu'))
tf_model.load_state_dict(saved_model)

train_dataset.transform = tf_model.tokenize
val_dataset.transform = tf_model.tokenize

In [7]:
out = Saliency(tf_model)

In [58]:
def get_attribution(model, dataset):
    model.eval()
    attrib_output = []

    for batch in dataset:
        labels = batch["target_values"]

        x_inp = batch["model_inputs"]["x"].requires_grad_(True)

        print(x_inp.shape)
        assert 1 == 0

        out = model(x_inp)
        loss = ((out - labels) ** 2).sum()
        loss.backward()

        attrib_output.append(x_inp.grad.abs().detach().cpu().numpy())

    return attrib_output

In [54]:
def return_normalized_and_aggregated_attribution(attrib_output):

    stacked_attrib = np.concatenate(attrib_output, axis=0)

    normed_list = []

    for i in range(stacked_attrib.shape[0]):
        normaliized_saliency = (stacked_attrib[i] - np.min(stacked_attrib[i]))
        normaliized_saliency = normaliized_saliency / np.max(normaliized_saliency)
        normed_list.append(normaliized_saliency)

    normed_attrib = np.stack(normed_list, axis=0)

    return normed_attrib

In [59]:
new_outs = get_attribution(tf_model, val_loader)

torch.Size([64, 100, 55])


AssertionError: 

In [56]:
normed_attrib = return_normalized_and_aggregated_attribution(new_outs)

In [57]:
normed_attrib.shape

(73, 100, 55)